In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install git+https://github.com/DAMO-NLP-SG/VideoLLaMA2.git --no-deps
!pip install git+https://github.com/openai/CLIP.git
!pip install decord
!pip install --upgrade pip torch torch_xla flash-attn --no-build-isolation

  Cloning https://github.com/DAMO-NLP-SG/VideoLLaMA2.git to /tmp/pip-req-build-kb31iapo
  Running command git clone --filter=blob:none --quiet https://github.com/DAMO-NLP-SG/VideoLLaMA2.git /tmp/pip-req-build-kb31iapo
  Resolved https://github.com/DAMO-NLP-SG/VideoLLaMA2.git to commit c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for videollama2: filename=videollama2-1.0-py3-none-any.whl size=5318463 sha256=5de20999eb774ee3107f7b54f015e7a8cc4e0b464f33ff0874ce7468e589b4be
  Stored in directory: /tmp/pip-ephem-wheel-cache-iaq_tcp4/wheels/7d/77/54/e9f72e919ccb645e74e2b527aa07e5fe3811a9705628e300b0
Successfully built videollama2
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-e2ueu5f2
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-e2ueu5f2
  Resolved https://github.c

In [ ]:
import os
import torch
import clip
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image
import subprocess
import csv
from collections import Counter
import warnings
from videollama2 import model_init, mm_infer
from videollama2.utils import disable_torch_init

/usr/local/lib/python3.11/dist-packages/torch_xla/__init__.py:251: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# Suppress unnecessary warnings from transformers
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

In [ ]:
os.environ["USE_FLASH_ATTENTION_2"] = "1"
os.environ["DISABLE_FLASH_ATTN"] = "0"
os.environ["HF_ATTENTION_IMPLEMENTATION"] = "flash_attention_2"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model = clip_model.float()  # Converting to FP32 for stability
print("CLIP model loaded.")

100%|███████████████████████████████████████| 338M/338M [00:12<00:00, 27.8MiB/s]


CLIP model loaded.


In [ ]:
def extract_frames(video_path, frame_rate=1):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % int(cap.get(cv2.CAP_PROP_FPS) // frame_rate) == 0:
            frames.append(frame)
        frame_count += 1

    cap.release()
    return frames

In [ ]:
def encode_frames(frames, model, preprocess, device):
    frame_tensors = [
        preprocess(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))).unsqueeze(0)
        for f in frames
    ]
    frame_tensors = torch.cat(frame_tensors).to(device)

    with torch.no_grad():
        frame_features = model.encode_image(frame_tensors).float()
        frame_features = frame_features.to(torch.float32)  # Convert explicitly to FP32
        frame_features /= frame_features.norm(dim=-1, keepdim=True)

    return frame_features.mean(dim=0, keepdim=True)  # Aggregate by mean pooling

In [ ]:
# Define paths for features and keyframe mappings
features_dir = "/content/drive/MyDrive/VQA_Dataset/clip-features-vit-b32-sample/clip-features"
map_dir = "/content/drive/MyDrive/VQA_Dataset/map-keyframes-sample/map-keyframes"

In [ ]:
# Load features and keyframe mappings
features = {file.split('.')[0]: torch.from_numpy(np.load(os.path.join(features_dir, file))).float().to(device)
            for file in os.listdir(features_dir) if file.endswith('.npy')}
keyframe_mappings = {file.split('.')[0]: list(csv.DictReader(open(os.path.join(map_dir, file), 'r')))
                     for file in os.listdir(map_dir) if file.endswith('.csv')}

In [ ]:
query = "Elephant in zoo"
print(f"Query: {query}")

# Encode text query
with torch.no_grad():
    text_features = clip_model.encode_text(clip.tokenize(query).to(device)).float()
    text_features /= text_features.norm(dim=-1, keepdim=True)
print("Query encoded")

Query: Elephant in zoo
Query encoded


In [ ]:
# Compute similarity scores
results = []
for video_id, video_features in tqdm(features.items()):
    similarities = (100.0 * video_features @ text_features.T).squeeze()
    top_indices = similarities.argsort(descending=True)[:5]
    for idx in top_indices:
        score = similarities[idx].item()
        frame_info = keyframe_mappings[video_id][idx]
        results.append((video_id, idx, frame_info['frame_idx'], frame_info['pts_time'], score))

# Sort results by similarity score
results.sort(key=lambda x: x[4], reverse=True)

100%|██████████| 31/31 [00:00<00:00, 413.91it/s]


In [ ]:
# Extract top 10 results
print("Top 10 results:")
for video_id, _, frame_idx, pts_time, score in results[:10]:
    print(f"Video ID: {video_id}, Frame Index: {frame_idx}, PTS Time: {pts_time}, Score: {score:.2f}")

# Identify most common video ID in top results
most_common_video_id, _ = Counter([video_id for video_id, _, _, _, _ in results[:10]]).most_common(1)[0]
print(f"Most common video ID: {most_common_video_id}")

# Extract timestamps for video clipping
pts_times = [float(pts_time) for video_id, _, _, pts_time, _ in results[:10] if video_id == most_common_video_id]
min_pts_time, max_pts_time = min(pts_times) - 3, max(pts_times) + 3

Top 10 results:
Video ID: L01_V016, Frame Index: 20486, PTS Time: 819.44, Score: 33.13
Video ID: L01_V016, Frame Index: 20339, PTS Time: 813.56, Score: 33.13
Video ID: L01_V016, Frame Index: 19764, PTS Time: 790.56, Score: 31.13
Video ID: L01_V014, Frame Index: 25287, PTS Time: 1011.48, Score: 30.33
Video ID: L01_V014, Frame Index: 25957, PTS Time: 1038.28, Score: 29.07
Video ID: L01_V024, Frame Index: 26280, PTS Time: 1051.2, Score: 29.02
Video ID: L01_V024, Frame Index: 26686, PTS Time: 1067.44, Score: 28.83
Video ID: L01_V016, Frame Index: 19925, PTS Time: 797.0, Score: 28.82
Video ID: L01_V015, Frame Index: 9560, PTS Time: 382.4, Score: 28.57
Video ID: L01_V024, Frame Index: 26430, PTS Time: 1057.2, Score: 28.56
Most common video ID: L01_V016


In [ ]:
# Define video paths
video_path = f'/content/drive/MyDrive/VQA_Dataset/video/{most_common_video_id}.mp4'
# Ensure video exists
if os.path.exists(video_path):
    print(f"Video found at: {video_path}")
else:
    print(f"Video NOT found at: {video_path}")

Video found at: /content/drive/MyDrive/VQA_Dataset/video/L01_V016.mp4


In [ ]:
from transformers import BitsAndBytesConfig
from videollama2 import model_init
import torch
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8-bit quantization
    llm_int8_threshold=6.0
)

model_path = "DAMO-NLP-SG/VideoLLaMA2-7B"  # Set your model path here

model, processor, tokenizer = model_init(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
instruct = "What is the color shirt is the woman wearing in the news deck?"
output = mm_infer(processor["video"](video_path), instruct, model=model, tokenizer=tokenizer, do_sample=False, modal="video")

print("Generated Answer:")
print(output)

Generated Answer:
The woman in the news deck is wearing a white shirt.
